In [1]:
import os
import re

In [13]:
import torch
from datasets import Dataset, load_dataset, concatenate_datasets, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, BitsAndBytesConfig, logging
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

In [3]:

import gc
gc.collect()
torch.cuda.empty_cache()

my_cache_dir = r"E:\Models"
model_name="Qwen/Qwen2.5-3B"

In [4]:
BLOCK_SIZE = 1024  
OVERLAP = 128
STRIDE = BLOCK_SIZE - OVERLAP

In [5]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",cache_dir=my_cache_dir)

In [6]:

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True, # Quantized to 8 bits
    llm_int8_threshold=0.6,
    llm_int8_has_fp16_weight=False
)

In [32]:

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=my_cache_dir,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [33]:
model = prepare_model_for_kbit_training(model)

In [19]:

data = []

In [23]:


train_datasets = []
test_datasets = []

# Define the exact features you want to keep
target_features = ['text', 'framework', 'purpose', 'task', 'source']

for f_dir in os.listdir("./chunks"):
    full_path = os.path.join("./chunks", f_dir)
    if os.path.isfile(full_path) and full_path.endswith((".json", ".jsonl")):
        ds_dict = load_dataset('json', data_files=full_path)
        train_ds = ds_dict['train']
        
        # If a dataset is missing a feature, add it with None or a default value
        for feature in target_features:
            if feature not in train_ds.column_names:
                train_ds = train_ds.add_column(feature, [None] * len(train_ds))
                
        # If it has extra features, remove them
        extra_cols = [col for col in train_ds.column_names if col not in target_features]
        if extra_cols:
            train_ds = train_ds.remove_columns(extra_cols)
            
        # Split into train/test
        split_ds = train_ds.train_test_split(test_size=0.1, seed=42)
        train_datasets.append(split_ds['train'])
        test_datasets.append(split_ds['test'])

# Now combine safely
final_train = concatenate_datasets(train_datasets)
final_test = concatenate_datasets(test_datasets)

combined_dataset_dict = DatasetDict({
    "train": final_train,
    "test": final_test
})

<function concatenate_datasets at 0x000001B985ADACB0>


In [26]:
tokenized_dataset = combined_dataset_dict.map(tokenize_fn, batched=True, remove_columns=combined_dataset_dict["train"].column_names)
tokenized_dataset

Map:   0%|          | 0/1744 [00:00<?, ? examples/s]

Map:   0%|          | 0/196 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1744
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 196
    })
})

In [37]:
training_args = TrainingArguments(
    output_dir="./queen-docs-lora",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    save_steps=100,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_total_limit=1,
    report_to="none",
    remove_unused_columns=False
)


In [38]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)


In [39]:
non_inst_model = get_peft_model(model, lora_config)

C:\Users\SUHAN PATRA\AppData\Roaming\Python\Python310\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
C:\Users\SUHAN PATRA\AppData\Roaming\Python\Python310\site-packages\peft\tuners\tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [40]:
trainer = Trainer(
    args=training_args,
    model=non_inst_model,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)

In [ ]:
trainer.train(resume_from_checkpoint=True)

Step,Training Loss
10,6.317353
20,5.358252
30,4.770834
40,3.586604
50,4.022591
60,4.117896
70,3.609038
80,3.608713
90,3.610243
100,3.603922


config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

C:\Users\SUHAN PATRA\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SUHAN PATRA\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\SUHAN PATRA\torch_intermediate\lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWa

KeyboardInterrupt: 

In [ ]:
# for framework, folder_path in TARGET_FOLDERS.items():
#     if not os.path.exists(folder_path):
#         print(f"{framework} folder does not exists")
#         continue
#     print(f"Starting {framework}...")
#     for folders in os.listdir(folder_path):
#         if '.' not in folders:
#             print(folders)
#     # for root, dirs, folders in os.walk(folder_path):
#     #     if len(folders):
#     #         allowed_folders = [f for f in folders if '.' not in f]
#     #         print(folders)
#     #         print(allowed_folders)
#     #     continue
#     #         # print(f"Available folders {folders}")
#     #         # print(f"Folder Directory {dirs}")
#     #         # print(root)
#     print()
    

In [ ]:
# for u,root,f in os.walk("./docs"):
#     print(root)

In [ ]:
# # from transformers import AutoTokenizer
# # import os
# import json

# # # Use the exact model repo to get the correct tokenizer for chunking
# # model_id = "Qwen/Qwen2.5-3B-Instruct"
# # tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=my_cache_dir)

# # Chunking configuration
# BLOCK_SIZE = 1024
# OVERLAP = 128
# STRIDE = BLOCK_SIZE - OVERLAP

# TARGET_FOLDERS = {
#     "Next.js": "./docs/next.js-canary",
#     "NestJS": "./docs/nest"
# }

# allowed_extensions = {".md", ".ts", ".js", ".tsx", ".jsx"}
# ignored_directories = {"node_modules", "dist", "build", ".git", "coverage", ".next"}

# output_chunks = []

# for framework, repo_folder in TARGET_FOLDERS.items():
#     if not os.path.exists(repo_folder):
#         print(f"Skipping {framework}: Path {repo_folder} not found.")
#         continue

#     print(f"Processing {framework} from {repo_folder}...")

#     for root, dirs, files in os.walk(repo_folder):
#         dirs[:] = [d for d in dirs if d not in ignored_directories]

#         for filename in files:
#             ext = os.path.splitext(filename)[1].lower()
#             if ext not in allowed_extensions:
#                 continue

#             file_path = os.path.join(root, filename)
#             rel_path = os.path.relpath(file_path, repo_folder)
#             file_type = "Documentation" if ext == ".md" else "Source Code"

#             try:
#                 with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
#                     content = f.read().strip()
#             except Exception as e:
#                 print(f"Error reading {file_path}: {e}")
#                 continue

#             if not content:
#                 continue

#             if ext == ".md":
#                 body = content
#             else:
#                 body = f"```{ext[1:]}\n{content}\n```"

#             header = (
#                 f"Framework: {framework}\n"
#                 f"File Path: {rel_path}\n"
#                 f"File Type: {file_type}\n\n"
#             )

#             full_text = header + body
#             # tokens = tokenizer.encode(full_text, add_special_tokens=False)

#             # for i in range(0, len(tokens), STRIDE):
#             #     chunk_tokens = tokens[i:i + BLOCK_SIZE]
#             #     chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
#             #     output_chunks.append(
                    # {
                    #     "text": chunk_text,
                    #     "framework": framework,
                    #     "source": rel_path,
                    #     "file_type": file_type,
                    # }
#             #     )

# output_file = "next_nest_uninstructioned_dataset.jsonl"
# with open(output_file, "w", encoding="utf-8") as f:
#     for chunk in output_chunks:
#         f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

# print(f"Success! Generated {len(output_chunks)} chunks in {output_file}")